# Train PWPA on eight datasets

Trains PWPA on each development fold of each dataset, on the same splits as PIE, with RFE choosing the priors.
The fit stops early on the same validation knockdowns as PIE, and the partners are the fold's other training knockdowns.
The same partner weights give the DE calls and fold changes.
Writes `predictions/pwpa/<dataset>/fold_<f>/pwpa_f<f>.parquet`, in the format of PIE's predictions, and `priors.json`, the chosen priors and their gates.
The `DATASETS` and `FOLDS` environment variables (comma-separated, default all) pick the folds to run, so that one Slurm array task can run one dataset.

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import torch

sys.path.append("../utils")
from paths import data_dir
from pie_scoring import load, responses, split, write_predictions

sys.path.append("../1.model")
import pwpa

torch.set_num_threads(int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count())))

In [ ]:
datasets = (
    pd.read_csv("../0.datasets/datasets.tsv", sep="\t")
    .query("used_in == '2.train_test' and dataset != 'pie_replogle_nadig'")
    .dataset.tolist()
)
datasets = [d for d in datasets if d in os.environ.get("DATASETS", ",".join(datasets)).split(",")]
folds = [int(f) for f in os.environ.get("FOLDS", "0,1,2,3").split(",")]
datasets, folds

In [ ]:
priors = pwpa.load_priors(data_dir("priors"))  # PIE's five genetic knowledge sources

In [ ]:
for dataset in datasets:
    d = load(data_dir("perturbseq", "processed", dataset, "preprocessed"))
    splits = pd.read_csv(f"../0.datasets/splits/{dataset}.tsv", sep="\t")
    contexts = np.unique(d.context)
    for fold in folds:
        s = split(splits, fold)
        rows = {part: [np.flatnonzero((d.context == c) & np.isin(d.pert, genes)) for c in contexts] for part, genes in s.items()}
        fit = [np.concatenate(r) for r in zip(rows["train"], rows["val"])]  # validation knockdowns stop the fit early
        best = pwpa.choose(pwpa.rfe(priors, [(d.pert[r], responses(d, r)) for r in fit], held_out=s["val"]))
        chosen = [priors[name] for name in best["priors"]]
        out = []
        for train, test in zip(rows["train"], rows["test"]):  # partners: the context's training knockdowns
            delta, W = pwpa.predict(chosen, best["scale"], best["gates"], d.pert[test], d.pert[train], responses(d, train))
            p_de, lfc = pwpa.predict_de(W, d.de[train], d.lfc[train], own=d.own[train], tested=d.tested[train].any(0))
            out.append((test, p_de, lfc, delta))
        folder = data_dir("predictions", "pwpa", dataset, f"fold_{fold}")
        write_predictions(folder / f"pwpa_f{fold}.parquet", d, *(np.concatenate(x) for x in zip(*out)))
        gates = dict(zip(best["priors"], best["gates"].mean(0).tolist()))
        (folder / "priors.json").write_text(json.dumps({"gates": gates}, indent=1) + "\n")
        print(dataset, fold, gates, flush=True)